In [1]:
# ============================================================
# PHARMAOPT
# NOTEBOOK 08 — RESOURCE ALLOCATION OPTIMIZATION
# ============================================================
#
# PURPOSE
# Build a constrained optimization engine that combines:
#
# 1. Pharmaceutical supply-risk signals
# 2. Geographic demand / utilization
# 3. Limited available inventory
# 4. Explicit operational constraints
#
# The goal is to recommend modeled inventory allocations that
# reduce unmet demand under supply constraints.
#
# IMPORTANT:
# This is a decision-support model. Recommendations are modeled
# scenarios and are not autonomous real-world distribution decisions.
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_DIR = Path.cwd()

print("PHARMAOPT — RESOURCE ALLOCATION OPTIMIZATION")
print("=" * 70)
print("Project directory:", PROJECT_DIR)

PHARMAOPT — RESOURCE ALLOCATION OPTIMIZATION
Project directory: C:\Users\jpiza\PharmaOpt


In [2]:
# ============================================================
# OPTIMIZATION ENVIRONMENT CHECK
# ============================================================

try:
    import ortools

    print("OR-Tools is installed.")
    print("Version:", ortools.__version__)

except ImportError:
    print("OR-Tools is NOT installed yet.")

OR-Tools is installed.
Version: 9.15.6755


In [3]:
# ============================================================
# LOCATE CMS PART D GEOGRAPHIC UTILIZATION DATA
# ============================================================

import zipfile

DOWNLOADS_DIR = Path.home() / "Downloads"

search_locations = [
    PROJECT_DIR,
    DOWNLOADS_DIR
]

cms_zip_candidates = []

for folder in search_locations:
    cms_zip_candidates.extend(
        folder.glob("Medicare Part D Prescribers - by Geography and Drug*.zip")
    )

print("CMS ZIP FILES FOUND")
print("=" * 70)

for i, file in enumerate(cms_zip_candidates, start=1):
    print(f"{i}. {file}")

if not cms_zip_candidates:
    raise FileNotFoundError(
        "CMS Part D geographic data ZIP was not found."
    )

CMS ZIP FILES FOUND
1. C:\Users\jpiza\PharmaOpt\Medicare Part D Prescribers - by Geography and Drug (1).zip
2. C:\Users\jpiza\PharmaOpt\Medicare Part D Prescribers - by Geography and Drug.zip


In [4]:
# ============================================================
# INSPECT 2024 CMS GEOGRAPHIC DATA
# ============================================================

cms_data_zip = None
cms_2024_csv = None

for zip_path in cms_zip_candidates:
    
    with zipfile.ZipFile(zip_path, "r") as z:
        
        csv_files = [
            name for name in z.namelist()
            if name.lower().endswith(".csv")
        ]
        
        matches_2024 = [
            name for name in csv_files
            if "DY24" in name
        ]
        
        if matches_2024:
            cms_data_zip = zip_path
            cms_2024_csv = matches_2024[0]
            break

if cms_data_zip is None:
    raise FileNotFoundError(
        "Could not locate the 2024 CMS geographic CSV."
    )

print("CMS DATA ZIP")
print(cms_data_zip)

print("\n2024 FILE")
print(cms_2024_csv)

with zipfile.ZipFile(cms_data_zip, "r") as z:
    with z.open(cms_2024_csv) as f:
        cms_2024_sample = pd.read_csv(
            f,
            nrows=10
        )

print("\nCOLUMNS")
print("=" * 70)

for column in cms_2024_sample.columns:
    print(column)

print("\nSAMPLE")
display(cms_2024_sample)

CMS DATA ZIP
C:\Users\jpiza\PharmaOpt\Medicare Part D Prescribers - by Geography and Drug.zip

2024 FILE
Medicare Part D Prescribers - by Geography and Drug/2024/MUP_DPR_RY26_P04_V10_DY24_Geo.csv

COLUMNS
Prscrbr_Geo_Lvl
Prscrbr_Geo_Cd
Prscrbr_Geo_Desc
Brnd_Name
Gnrc_Name
Tot_Prscrbrs
Tot_Clms
Tot_30day_Fills
Tot_Drug_Cst
Tot_Benes
GE65_Sprsn_Flag
GE65_Tot_Clms
GE65_Tot_30day_Fills
GE65_Tot_Drug_Cst
GE65_Bene_Sprsn_Flag
GE65_Tot_Benes
LIS_Bene_Cst_Shr
NonLIS_Bene_Cst_Shr
Opioid_Drug_Flag
Opioid_LA_Drug_Flag
Antbtc_Drug_Flag
Antpsyct_Drug_Flag

SAMPLE


,Prscrbr_Geo_Lvl,Prscrbr_Geo_Cd,Prscrbr_Geo_Desc,Brnd_Name,Gnrc_Name,Tot_Prscrbrs,Tot_Clms,Tot_30day_Fills,Tot_Drug_Cst,Tot_Benes,...,GE65_Tot_30day_Fills,GE65_Tot_Drug_Cst,GE65_Bene_Sprsn_Flag,GE65_Tot_Benes,LIS_Bene_Cst_Shr,NonLIS_Bene_Cst_Shr,Opioid_Drug_Flag,Opioid_LA_Drug_Flag,Antbtc_Drug_Flag,Antpsyct_Drug_Flag
0,National,NaN,National,1st Tier Unifine Pentips,"Pen Needle, Diabetic",209,378,695.6,1.045460e+04,213,...,584.4,8.336360e+03,NaN,174.0,140.44,3053.27,N,N,N,N
1,National,NaN,National,1st Tier Unifine Pentips Plus,"Pen Needle, Diabetic",442,965,1588.4,2.607456e+04,451,...,1301.4,2.091875e+04,NaN,360.0,686.07,5152.25,N,N,N,N
2,National,NaN,National,Abacavir,Abacavir Sulfate,2277,16298,21345.5,3.054440e+06,2349,...,14073.4,1.994516e+06,NaN,1491.0,6378.08,106462.22,N,N,N,N
3,National,NaN,National,Abacavir-Lamivudine,Abacavir Sulfate/Lamivudine,2162,17348,24469.2,5.785644e+06,2467,...,16985.9,3.873673e+06,NaN,1708.0,8498.40,266361.31,N,N,N,N
4,National,NaN,National,Abilify,Aripiprazole,1715,7131,9532.7,7.066614e+06,1478,...,3108.4,2.038721e+06,NaN,536.0,10911.96,119541.25,N,N,N,Y
5,National,NaN,National,Abilify Asimtufii,Aripiprazole,2505,11620,21637.4,6.313229e+07,3360,...,2748.7,7.899070e+06,NaN,417.0,16712.19,42861.53,N,N,N,Y
6,National,NaN,National,Abilify Maintena,Aripiprazole,19332,234939,236211.6,6.434650e+08,29870,...,40015.1,1.070583e+08,NaN,5015.0,359918.88,998409.33,N,N,N,Y
7,National,NaN,National,Abilify Mycite,Aripiprazole,11,53,53.0,9.032238e+04,16,...,NaN,NaN,*,NaN,99.40,100.00,N,N,N,Y
8,National,NaN,National,Abiraterone Acetate,Abiraterone Acetate,14008,389330,407947.4,6.928357e+08,55268,...,396606.9,6.713460e+08,NaN,53648.0,818292.48,64497380.63,N,N,N,N
9,National,NaN,National,Abraxane,Paclitaxel Protein-Bound,154,984,984.0,7.163424e+06,234,...,902.0,6.575312e+06,NaN,212.0,3848.44,146595.17,N,N,N,N


In [5]:
# ============================================================
# LOAD FULL 2024 CMS DATA + INSPECT GEOGRAPHY STRUCTURE
# ============================================================

with zipfile.ZipFile(cms_data_zip, "r") as z:
    with z.open(cms_2024_csv) as f:
        cms_2024 = pd.read_csv(f)

print("2024 CMS DATA LOADED")
print("=" * 70)

print(f"Rows:    {len(cms_2024):,}")
print(f"Columns: {len(cms_2024.columns):,}")

print("\nALL COLUMNS")
print("=" * 70)

for i, column in enumerate(cms_2024.columns, start=1):
    print(f"{i:>2}. {column}")

geo_columns = [
    column
    for column in cms_2024.columns
    if "geo" in column.lower()
]

print("\nPOSSIBLE GEOGRAPHY COLUMNS")
print("=" * 70)

for column in geo_columns:
    print(column)

2024 CMS DATA LOADED
Rows:    117,661
Columns: 22

ALL COLUMNS
 1. Prscrbr_Geo_Lvl
 2. Prscrbr_Geo_Cd
 3. Prscrbr_Geo_Desc
 4. Brnd_Name
 5. Gnrc_Name
 6. Tot_Prscrbrs
 7. Tot_Clms
 8. Tot_30day_Fills
 9. Tot_Drug_Cst
10. Tot_Benes
11. GE65_Sprsn_Flag
12. GE65_Tot_Clms
13. GE65_Tot_30day_Fills
14. GE65_Tot_Drug_Cst
15. GE65_Bene_Sprsn_Flag
16. GE65_Tot_Benes
17. LIS_Bene_Cst_Shr
18. NonLIS_Bene_Cst_Shr
19. Opioid_Drug_Flag
20. Opioid_LA_Drug_Flag
21. Antbtc_Drug_Flag
22. Antpsyct_Drug_Flag

POSSIBLE GEOGRAPHY COLUMNS
Prscrbr_Geo_Lvl
Prscrbr_Geo_Cd
Prscrbr_Geo_Desc


C:\Users\jpiza\AppData\Local\Temp\ipykernel_58628\752958549.py:7: DtypeWarning: Columns (1: Prscrbr_Geo_Cd) have mixed types. Specify dtype option on import or set low_memory=False.
  cms_2024 = pd.read_csv(f)


In [6]:
# ============================================================
# INSPECT CMS GEOGRAPHIC LEVELS
# ============================================================

print("GEOGRAPHY COLUMN VALUES")
print("=" * 70)

for column in geo_columns:
    
    print(f"\nCOLUMN: {column}")
    print("-" * 70)
    
    print(
        cms_2024[column]
        .value_counts(dropna=False)
        .head(25)
    )

GEOGRAPHY COLUMN VALUES

COLUMN: Prscrbr_Geo_Lvl
----------------------------------------------------------------------
Prscrbr_Geo_Lvl
State       114029
National      3632
Name: count, dtype: int64

COLUMN: Prscrbr_Geo_Cd
----------------------------------------------------------------------
Prscrbr_Geo_Cd
NaN     3633
36      2980
6.0     2979
12.0    2882
42      2760
34      2642
39      2626
37      2579
26      2547
17.0    2542
13.0    2517
25      2474
47      2414
51      2406
24      2382
4.0     2358
29      2343
18      2333
21      2324
9.0     2304
22      2294
1.0     2287
45      2270
53      2261
55      2243
Name: count, dtype: int64

COLUMN: Prscrbr_Geo_Desc
----------------------------------------------------------------------
Prscrbr_Geo_Desc
National          3632
New York          2980
California        2979
Florida           2882
Texas             2794
Pennsylvania      2760
New Jersey        2642
Ohio              2626
North Carolina    2579
Michigan          

In [7]:
# ============================================================
# BUILD 2024 STATE-LEVEL CMS DEMAND TABLE
# ============================================================

state_columns = [
    "Prscrbr_Geo_Desc",
    "Brnd_Name",
    "Gnrc_Name",
    "Tot_Prscrbrs",
    "Tot_Clms",
    "Tot_30day_Fills",
    "Tot_Benes"
]

cms_state_2024 = (
    cms_2024.loc[
        cms_2024["Prscrbr_Geo_Lvl"].eq("State"),
        state_columns
    ]
    .copy()
)

# Make sure utilization measures are numeric
numeric_columns = [
    "Tot_Prscrbrs",
    "Tot_Clms",
    "Tot_30day_Fills",
    "Tot_Benes"
]

for column in numeric_columns:
    cms_state_2024[column] = pd.to_numeric(
        cms_state_2024[column],
        errors="coerce"
    )

# Rename fields to clearer PharmaOpt names
cms_state_2024 = cms_state_2024.rename(
    columns={
        "Prscrbr_Geo_Desc": "state",
        "Brnd_Name": "brand_name",
        "Gnrc_Name": "generic_name",
        "Tot_Prscrbrs": "total_prescribers",
        "Tot_Clms": "total_claims",
        "Tot_30day_Fills": "total_30day_fills",
        "Tot_Benes": "total_beneficiaries"
    }
)

print("PHARMAOPT — 2024 STATE DEMAND TABLE")
print("=" * 70)

print(f"Rows: {len(cms_state_2024):,}")
print(
    f"Geographic markets: "
    f"{cms_state_2024['state'].nunique():,}"
)

print(
    f"Unique generic drugs: "
    f"{cms_state_2024['generic_name'].nunique():,}"
)

print(
    f"Unique brand drugs: "
    f"{cms_state_2024['brand_name'].nunique():,}"
)

print(
    f"Missing 30-day fills: "
    f"{cms_state_2024['total_30day_fills'].isna().sum():,}"
)

# Check whether a drug/state combination appears more than once
duplicate_count = cms_state_2024.duplicated(
    subset=[
        "state",
        "brand_name",
        "generic_name"
    ]
).sum()

print(
    f"Duplicate state/brand/generic combinations: "
    f"{duplicate_count:,}"
)

print("\nGEOGRAPHIC MARKETS")
print("=" * 70)

print(
    sorted(
        cms_state_2024["state"]
        .dropna()
        .unique()
    )
)

print("\nSAMPLE")
display(cms_state_2024.head(10))

PHARMAOPT — 2024 STATE DEMAND TABLE
Rows: 114,029
Geographic markets: 60
Unique generic drugs: 1,911
Unique brand drugs: 3,289
Missing 30-day fills: 0
Duplicate state/brand/generic combinations: 0

GEOGRAPHIC MARKETS
['Alabama', 'Alaska', 'Arizona', 'Arkansas', 'Armed Forces Central/South America', 'Armed Forces Europe', 'Armed Forces Pacific', 'California', 'Colorado', 'Connecticut', 'Delaware', 'District of Columbia', 'Florida', 'Foreign Country', 'Georgia', 'Guam', 'Hawaii', 'Idaho', 'Illinois', 'Indiana', 'Iowa', 'Kansas', 'Kentucky', 'Louisiana', 'Maine', 'Maryland', 'Massachusetts', 'Michigan', 'Minnesota', 'Mississippi', 'Missouri', 'Montana', 'Nebraska', 'Nevada', 'New Hampshire', 'New Jersey', 'New Mexico', 'New York', 'North Carolina', 'North Dakota', 'Northern Mariana Islands', 'Ohio', 'Oklahoma', 'Oregon', 'Pennsylvania', 'Puerto Rico', 'Rhode Island', 'South Carolina', 'South Dakota', 'Tennessee', 'Texas', 'Unknown', 'Utah', 'Vermont', 'Virgin Islands', 'Virginia', 'Washin

,state,brand_name,generic_name,total_prescribers,total_claims,total_30day_fills,total_beneficiaries
3632,NaN,Trazodone Hcl,Trazodone Hcl,1,13,13.0,13.0
3633,Alabama,1st Tier Unifine Pentips Plus,"Pen Needle, Diabetic",7,11,27.7,NaN
3634,Alabama,Abacavir,Abacavir Sulfate,36,273,303.1,41.0
3635,Alabama,Abacavir-Lamivudine,Abacavir Sulfate/Lamivudine,18,117,157.0,21.0
3636,Alabama,Abilify,Aripiprazole,12,34,49.4,NaN
3637,Alabama,Abilify Asimtufii,Aripiprazole,43,261,481.5,72.0
3638,Alabama,Abilify Maintena,Aripiprazole,288,5935,5943.5,741.0
3639,Alabama,Abiraterone Acetate,Abiraterone Acetate,184,3406,3504.0,567.0
3640,Alabama,Abraxane,Paclitaxel Protein-Bound,3,25,25.0,NaN
3641,Alabama,Abrysvo,"Rsv Vacc, Pref A And Pref B/Pf",764,17737,17740.0,17668.0


In [8]:
# ============================================================
# PHARMAOPT — CLEAN U.S. STATE + DC DEMAND TABLE
# ============================================================

US_MARKETS = [
    "Alabama", "Alaska", "Arizona", "Arkansas", "California",
    "Colorado", "Connecticut", "Delaware", "District of Columbia",
    "Florida", "Georgia", "Hawaii", "Idaho", "Illinois", "Indiana",
    "Iowa", "Kansas", "Kentucky", "Louisiana", "Maine", "Maryland",
    "Massachusetts", "Michigan", "Minnesota", "Mississippi",
    "Missouri", "Montana", "Nebraska", "Nevada", "New Hampshire",
    "New Jersey", "New Mexico", "New York", "North Carolina",
    "North Dakota", "Ohio", "Oklahoma", "Oregon", "Pennsylvania",
    "Rhode Island", "South Carolina", "South Dakota", "Tennessee",
    "Texas", "Utah", "Vermont", "Virginia", "Washington",
    "West Virginia", "Wisconsin", "Wyoming"
]

# Keep only the 50 states + Washington, DC
cms_us_2024 = cms_state_2024[
    cms_state_2024["state"].isin(US_MARKETS)
].copy()

# Aggregate multiple brand products belonging to the same generic drug
generic_state_demand_2024 = (
    cms_us_2024
    .groupby(
        ["generic_name", "state"],
        as_index=False
    )
    .agg(
        total_30day_fills=("total_30day_fills", "sum"),
        total_claims=("total_claims", "sum"),
        brand_count=("brand_name", "nunique")
    )
)

print("PHARMAOPT — CLEAN 2024 GENERIC-STATE DEMAND")
print("=" * 75)

print(f"Original state-level rows: {len(cms_state_2024):,}")
print(f"Rows retained in 50 states + DC: {len(cms_us_2024):,}")
print(f"Geographic markets retained: {cms_us_2024['state'].nunique():,}")
print(
    f"Generic-state combinations: "
    f"{len(generic_state_demand_2024):,}"
)
print(
    f"Unique generic drugs: "
    f"{generic_state_demand_2024['generic_name'].nunique():,}"
)

print("\nEXCLUDED CMS GEOGRAPHIES")
print("=" * 75)

excluded_geographies = sorted(
    cms_state_2024.loc[
        ~cms_state_2024["state"].isin(US_MARKETS),
        "state"
    ]
    .dropna()
    .unique()
)

for geography in excluded_geographies:
    print(geography)

print("\nSAMPLE GENERIC-STATE DEMAND")
display(generic_state_demand_2024.head(15))

PHARMAOPT — CLEAN 2024 GENERIC-STATE DEMAND
Original state-level rows: 114,029
Rows retained in 50 states + DC: 109,519
Geographic markets retained: 51
Generic-state combinations: 68,698
Unique generic drugs: 1,908

EXCLUDED CMS GEOGRAPHIES
Armed Forces Central/South America
Armed Forces Europe
Armed Forces Pacific
Foreign Country
Guam
Northern Mariana Islands
Puerto Rico
Unknown
Virgin Islands

SAMPLE GENERIC-STATE DEMAND


,generic_name,state,total_30day_fills,total_claims,brand_count
0,0.9 % Sodium Chloride,Alabama,9741.0,9571,1
1,0.9 % Sodium Chloride,Alaska,105.9,103,1
2,0.9 % Sodium Chloride,Arizona,5528.0,5420,1
3,0.9 % Sodium Chloride,Arkansas,6172.9,5922,1
4,0.9 % Sodium Chloride,California,42784.6,41632,1
5,0.9 % Sodium Chloride,Colorado,2893.7,2853,1
6,0.9 % Sodium Chloride,Connecticut,2660.9,2629,1
7,0.9 % Sodium Chloride,Delaware,1116.1,1089,1
8,0.9 % Sodium Chloride,District of Columbia,534.2,517,1
9,0.9 % Sodium Chloride,Florida,26887.1,26389,1


In [9]:
# ============================================================
# PHARMAOPT — CONNECT FDA SHORTAGES TO CMS DEMAND
# ============================================================

import json
import zipfile
import re
import unicodedata


# ------------------------------------------------------------
# DRUG NAME NORMALIZATION
# ------------------------------------------------------------

def normalize_drug_name(value):

    if pd.isna(value):
        return None

    value = str(value)

    value = unicodedata.normalize(
        "NFKD",
        value
    )

    value = value.lower()

    value = re.sub(
        r"[^a-z0-9]+",
        " ",
        value
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    ).strip()

    return value


# ------------------------------------------------------------
# LOAD FDA SHORTAGE DATA
# ------------------------------------------------------------

shortage_zip = (
    PROJECT_DIR /
    "drug-shortages-0001-of-0001.json.zip"
)

with zipfile.ZipFile(shortage_zip, "r") as z:

    json_file = z.namelist()[0]

    with z.open(json_file) as f:
        shortage_json = json.load(f)


shortage_df = pd.json_normalize(
    shortage_json["results"]
)


print("FDA shortage rows:", f"{len(shortage_df):,}")


# ------------------------------------------------------------
# NORMALIZE FDA + CMS GENERIC NAMES
# ------------------------------------------------------------

shortage_df["generic_normalized"] = (
    shortage_df["generic_name"]
    .apply(normalize_drug_name)
)


generic_state_demand_2024["generic_normalized"] = (
    generic_state_demand_2024["generic_name"]
    .apply(normalize_drug_name)
)


# ------------------------------------------------------------
# FDA SHORTAGE GENERIC SET
# ------------------------------------------------------------

shortage_generics = set(
    shortage_df[
        "generic_normalized"
    ].dropna()
)


cms_shortage_overlap = (
    generic_state_demand_2024[
        generic_state_demand_2024[
            "generic_normalized"
        ].isin(shortage_generics)
    ]
    .copy()
)


print("\nFDA ↔ CMS OVERLAP")
print("=" * 75)

print(
    "FDA shortage generic names:",
    f"{len(shortage_generics):,}"
)

print(
    "CMS generic drugs with FDA shortage history:",
    f"{cms_shortage_overlap['generic_normalized'].nunique():,}"
)

print(
    "Drug-state observations:",
    f"{len(cms_shortage_overlap):,}"
)

FDA shortage rows: 1,601

FDA ↔ CMS OVERLAP
FDA shortage generic names: 241
CMS generic drugs with FDA shortage history: 0
Drug-state observations: 0


In [10]:
# ============================================================
# PHARMAOPT — IDENTIFY ALLOCATION SCENARIO CANDIDATES
# ============================================================

allocation_candidates = (
    cms_shortage_overlap
    .groupby(
        [
            "generic_normalized",
            "generic_name"
        ],
        as_index=False
    )
    .agg(

        states_with_demand=(
            "state",
            "nunique"
        ),

        national_30day_fills_proxy=(
            "total_30day_fills",
            "sum"
        ),

        national_claims_proxy=(
            "total_claims",
            "sum"
        )
    )
)


# Prefer drugs with broad geographic coverage
allocation_candidates = (
    allocation_candidates[
        allocation_candidates[
            "states_with_demand"
        ] >= 40
    ]
    .sort_values(
        "national_30day_fills_proxy",
        ascending=False
    )
    .reset_index(drop=True)
)


print("PHARMAOPT ALLOCATION CANDIDATES")
print("=" * 75)

print(
    "Candidates with demand in at least 40 markets:",
    f"{len(allocation_candidates):,}"
)

display(
    allocation_candidates.head(25)
)

PHARMAOPT ALLOCATION CANDIDATES
Candidates with demand in at least 40 markets: 0


,generic_normalized,generic_name,states_with_demand,national_30day_fills_proxy,national_claims_proxy


In [11]:
# ============================================================
# PHARMAOPT — AUDIT FDA ↔ CMS DRUG MATCH COVERAGE
# ============================================================

print("FDA ↔ CMS MATCH AUDIT")
print("=" * 75)

print(
    "Unique FDA shortage generic names:",
    f"{len(shortage_generics):,}"
)

print(
    "Exact CMS/FDA matched generic drugs:",
    f"{cms_shortage_overlap['generic_normalized'].nunique():,}"
)

print(
    "Matched drug-state observations:",
    f"{len(cms_shortage_overlap):,}"
)


# ------------------------------------------------------------
# CHECK GEOGRAPHIC COVERAGE OF EXACT MATCHES
# ------------------------------------------------------------

exact_match_coverage = (
    cms_shortage_overlap
    .groupby(
        ["generic_normalized", "generic_name"],
        as_index=False
    )
    .agg(
        states_with_demand=("state", "nunique"),
        total_30day_fills=("total_30day_fills", "sum"),
        total_claims=("total_claims", "sum")
    )
    .sort_values(
        ["states_with_demand", "total_30day_fills"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

if len(exact_match_coverage) > 0:

    print(
        "\nMaximum geographic coverage:",
        exact_match_coverage["states_with_demand"].max()
    )

    print("\nTOP EXACT MATCHES")
    print("=" * 75)

    display(
        exact_match_coverage.head(30)
    )

else:

    print("\nNo exact normalized generic-name matches were found.")


# ------------------------------------------------------------
# SHOW SAMPLE SOURCE NAMES
# ------------------------------------------------------------

print("\nSAMPLE FDA SHORTAGE GENERIC NAMES")
print("=" * 75)

display(
    shortage_df[
        ["generic_name", "generic_normalized"]
    ]
    .drop_duplicates()
    .head(20)
)


print("\nSAMPLE CMS GENERIC NAMES")
print("=" * 75)

display(
    generic_state_demand_2024[
        ["generic_name", "generic_normalized"]
    ]
    .drop_duplicates()
    .head(20)
)

FDA ↔ CMS MATCH AUDIT
Unique FDA shortage generic names: 241
Exact CMS/FDA matched generic drugs: 0
Matched drug-state observations: 0

No exact normalized generic-name matches were found.

SAMPLE FDA SHORTAGE GENERIC NAMES


,generic_name,generic_normalized
0,Midazolam Hydrochloride Injection,midazolam hydrochloride injection
1,Quinapril Hydrochloride Tablet,quinapril hydrochloride tablet
2,Furosemide Injection,furosemide injection
3,Ropivacaine Hydrochloride Injection,ropivacaine hydrochloride injection
4,Clindamycin Hydrochloride Capsule,clindamycin hydrochloride capsule
5,Dexmedetomidine Hydrochloride Injection,dexmedetomidine hydrochloride injection
7,"Methylphenidate Hydrochloride Tablet, Extended...",methylphenidate hydrochloride tablet extended ...
8,Frovatriptan Succinate Tablet,frovatriptan succinate tablet
9,Belladonna and Opium Suppository,belladonna and opium suppository
10,Morphine Sulfate Injection,morphine sulfate injection



SAMPLE CMS GENERIC NAMES


,generic_name,generic_normalized
0,0.9 % Sodium Chloride,0 9 sodium chloride
51,Aa 4.25 % No.1/Dextrose 5 %,aa 4 25 no 1 dextrose 5
54,Aa 5 % No.6/D15w/Lytes No.23,aa 5 no 6 d15w lytes no 23
55,Aa 5 % No.6/D20w/Lytes No.23,aa 5 no 6 d20w lytes no 23
57,Abacavir Sulfate,abacavir sulfate
104,Abacavir Sulfate/Lamivudine,abacavir sulfate lamivudine
150,Abacavir/Dolutegravir/Lamivudi,abacavir dolutegravir lamivudi
201,Abacavir/Lamivudine/Zidovudine,abacavir lamivudine zidovudine
203,Abaloparatide,abaloparatide
254,Abatacept,abatacept


In [12]:
# ============================================================
# PHARMAOPT — CONNECT TO MYSQL
# ============================================================

import mysql.connector
import getpass

mysql_password = getpass.getpass(
    "Enter your MySQL password: "
)

connection = mysql.connector.connect(
    host="127.0.0.1",
    port=3306,
    user="root",
    password=mysql_password,
    database="pharmaopt"
)

print("Connected to PharmaOpt MySQL database.")

Enter your MySQL password:  ········


Connected to PharmaOpt MySQL database.


In [13]:
# ============================================================
# PHARMAOPT — SHORTAGE → NDC PRODUCT BRIDGE
# ============================================================

def query_df(sql, params=None):

    cursor = connection.cursor()

    cursor.execute(
        sql,
        params or ()
    )

    rows = cursor.fetchall()

    columns = [
        description[0]
        for description in cursor.description
    ]

    cursor.close()

    return pd.DataFrame(
        rows,
        columns=columns
    )


shortage_ndc_products = query_df(
    """
    SELECT DISTINCT
        s.shortage_id,
        s.generic_name AS shortage_generic_name,
        s.linked_package_ndc,

        p.product_id,
        p.product_ndc,
        p.generic_name AS ndc_generic_name,
        p.brand_name,
        p.dosage_form,
        p.labeler_name

    FROM shortage_event AS s

    INNER JOIN drug_package AS pkg
        ON s.linked_package_ndc = pkg.package_ndc

    INNER JOIN drug_product AS p
        ON pkg.product_id = p.product_id

    WHERE s.linked_package_ndc IS NOT NULL
    """
)


print("SHORTAGE → NDC PRODUCT BRIDGE")
print("=" * 75)

print(
    "Linked shortage records:",
    f"{len(shortage_ndc_products):,}"
)

print(
    "Unique linked package NDCs:",
    f"{shortage_ndc_products['linked_package_ndc'].nunique():,}"
)

print(
    "Unique NDC generic names:",
    f"{shortage_ndc_products['ndc_generic_name'].nunique():,}"
)

print("\nSAMPLE")
display(
    shortage_ndc_products[
        [
            "shortage_generic_name",
            "ndc_generic_name",
            "brand_name",
            "dosage_form"
        ]
    ]
    .drop_duplicates()
    .head(25)
)

SHORTAGE → NDC PRODUCT BRIDGE
Linked shortage records: 1,421
Unique linked package NDCs: 1,393
Unique NDC generic names: 320

SAMPLE


,shortage_generic_name,ndc_generic_name,brand_name,dosage_form
0,Midazolam Hydrochloride Injection,MIDAZOLAM HYDROCHLORIDE,Midazolam,"INJECTION, SOLUTION"
1,Quinapril Hydrochloride Tablet,Quinapril,Quinapril,"TABLET, FILM COATED"
2,Furosemide Injection,Furosemide,Furosemide,"INJECTION, SOLUTION"
3,Ropivacaine Hydrochloride Injection,ROPIVACAINE HYDROCHLORIDE,Naropin,"INJECTION, SOLUTION"
4,Clindamycin Hydrochloride Capsule,clindamycin hydrochloride,Cleocin Hydrochloride,CAPSULE
5,Dexmedetomidine Hydrochloride Injection,DEXMEDETOMIDINE HYDROCHLORIDE,Precedex,"INJECTION, SOLUTION"
7,"Methylphenidate Hydrochloride Tablet, Extended...",methylphenidate hydrochloride,RELEXXII,"TABLET, EXTENDED RELEASE"
8,Belladonna and Opium Suppository,ATROPA BELLADONNA and OPIUM,Belladonna and Opium,SUPPOSITORY
9,Morphine Sulfate Injection,Morphine Sulfate,MITIGO,INJECTION
10,Etomidate Injection,Etomidate,Etomidate,INJECTION


In [14]:
# ============================================================
# PHARMAOPT — AUDIT NDC GENERIC ↔ CMS GENERIC MATCHING
# ============================================================

shortage_ndc_products[
    "ndc_generic_normalized"
] = (
    shortage_ndc_products[
        "ndc_generic_name"
    ]
    .apply(normalize_drug_name)
)


cms_ndc_overlap = (
    generic_state_demand_2024.merge(
        shortage_ndc_products[
            [
                "ndc_generic_name",
                "ndc_generic_normalized"
            ]
        ]
        .drop_duplicates(),

        left_on="generic_normalized",
        right_on="ndc_generic_normalized",

        how="inner"
    )
)


print("FDA NDC ↔ CMS MATCH AUDIT")
print("=" * 75)

print(
    "Unique shortage-linked NDC generic names:",
    f"{shortage_ndc_products['ndc_generic_normalized'].nunique():,}"
)

print(
    "Exact CMS/NDC matched generic drugs:",
    f"{cms_ndc_overlap['generic_normalized'].nunique():,}"
)

print(
    "Matched drug-state observations:",
    f"{len(cms_ndc_overlap):,}"
)


if len(cms_ndc_overlap) > 0:

    ndc_match_coverage = (
        cms_ndc_overlap
        .groupby(
            [
                "generic_normalized",
                "generic_name"
            ],
            as_index=False
        )
        .agg(
            states_with_demand=(
                "state",
                "nunique"
            ),

            total_30day_fills=(
                "total_30day_fills",
                "sum"
            ),

            total_claims=(
                "total_claims",
                "sum"
            )
        )
        .sort_values(
            [
                "states_with_demand",
                "total_30day_fills"
            ],
            ascending=[
                False,
                False
            ]
        )
        .reset_index(drop=True)
    )

    print(
        "\nMaximum geographic coverage:",
        ndc_match_coverage[
            "states_with_demand"
        ].max()
    )

    print("\nTOP MATCHED DRUGS")
    print("=" * 75)

    display(
        ndc_match_coverage.head(30)
    )

else:

    print(
        "\nStill no exact matches — "
        "additional ingredient normalization will be required."
    )

FDA NDC ↔ CMS MATCH AUDIT
Unique shortage-linked NDC generic names: 240
Exact CMS/NDC matched generic drugs: 107
Matched drug-state observations: 5,715

Maximum geographic coverage: 51

TOP MATCHED DRUGS


,generic_normalized,generic_name,states_with_demand,total_30day_fills,total_claims
0,furosemide,Furosemide,51,132984089.1,69973389
1,metoprolol succinate,Metoprolol Succinate,51,75953590.3,31714694
2,omeprazole,Omeprazole,51,63632501.0,28344703
3,hydrochlorothiazide,Hydrochlorothiazide,51,45875396.3,18114858
4,potassium chloride,Potassium Chloride,51,29802722.5,15738303
5,montelukast sodium,Montelukast Sodium,51,27062068.2,12090298
6,albuterol sulfate,Albuterol Sulfate,51,25008428.8,20974090
7,allopurinol,Allopurinol,51,22959134.2,9582901
8,spironolactone,Spironolactone,51,19953579.2,8981297
9,clonazepam,Clonazepam,51,18154200.2,16405904


In [15]:
# ============================================================
# PHARMAOPT — AUDIT ALLOCATION SCENARIO CANDIDATES
# ============================================================

# Exact generic names successfully linked between FDA NDC and CMS
matched_generic_names = set(
    cms_ndc_overlap[
        "generic_normalized"
    ].dropna()
)

# Keep shortage-linked NDC products that also match CMS
candidate_supply_records = (
    shortage_ndc_products[
        shortage_ndc_products[
            "ndc_generic_normalized"
        ].isin(matched_generic_names)
    ]
    .copy()
)

# Pull shortage-event details from MySQL
shortage_details = query_df(
    """
    SELECT
        shortage_id,
        generic_name AS shortage_generic_name,
        status,
        shortage_reason,
        therapeutic_category,
        dosage_form,
        initial_posting_date,
        update_date,
        linked_package_ndc
    FROM shortage_event
    WHERE linked_package_ndc IS NOT NULL
    """
)

# Connect shortage-event details to NDC generic ingredients
candidate_audit = (
    candidate_supply_records[
        [
            "shortage_id",
            "ndc_generic_name",
            "ndc_generic_normalized",
            "dosage_form"
        ]
    ]
    .rename(
        columns={
            "dosage_form": "ndc_dosage_form"
        }
    )
    .merge(
        shortage_details,
        on="shortage_id",
        how="left"
    )
)

# Add CMS geographic-demand coverage
candidate_audit = candidate_audit.merge(
    ndc_match_coverage[
        [
            "generic_normalized",
            "states_with_demand",
            "total_30day_fills",
            "total_claims"
        ]
    ],
    left_on="ndc_generic_normalized",
    right_on="generic_normalized",
    how="left"
)

print("PHARMAOPT — ALLOCATION CANDIDATE AUDIT")
print("=" * 80)

print(
    "Matched shortage records:",
    f"{len(candidate_audit):,}"
)

print(
    "Unique candidate generic drugs:",
    f"{candidate_audit['ndc_generic_normalized'].nunique():,}"
)

print("\nTOP CANDIDATES WITH FDA SHORTAGE CONTEXT")
print("=" * 80)

candidate_summary = (
    candidate_audit
    .groupby(
        [
            "ndc_generic_normalized",
            "ndc_generic_name"
        ],
        as_index=False
    )
    .agg(
        shortage_records=("shortage_id", "nunique"),
        shortage_descriptions=(
            "shortage_generic_name",
            lambda x: " | ".join(
                sorted(set(x.dropna().astype(str)))
            )
        ),
        ndc_dosage_forms=(
            "ndc_dosage_form",
            lambda x: " | ".join(
                sorted(set(x.dropna().astype(str)))
            )
        ),
        states_with_demand=("states_with_demand", "max"),
        total_30day_fills=("total_30day_fills", "max"),
        total_claims=("total_claims", "max")
    )
    .sort_values(
        [
            "states_with_demand",
            "total_30day_fills"
        ],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

display(
    candidate_summary.head(30)
)

PHARMAOPT — ALLOCATION CANDIDATE AUDIT
Matched shortage records: 617
Unique candidate generic drugs: 107

TOP CANDIDATES WITH FDA SHORTAGE CONTEXT


,ndc_generic_normalized,ndc_generic_name,shortage_records,shortage_descriptions,ndc_dosage_forms,states_with_demand,total_30day_fills,total_claims
0,furosemide,FUROSEMIDE,6,Furosemide Injection,"INJECTION, SOLUTION",51,132984089.1,69973389
1,furosemide,Furosemide,20,Furosemide Injection | Furosemide Oral Solution,"INJECTION | INJECTION, SOLUTION | SOLUTION",51,132984089.1,69973389
2,furosemide,furosemide,6,Furosemide Injection,"INJECTION, SOLUTION",51,132984089.1,69973389
3,metoprolol succinate,metoprolol succinate,7,"Metoprolol Succinate Tablet, Extended Release","TABLET, FILM COATED, EXTENDED RELEASE",51,75953590.3,31714694
4,omeprazole,Omeprazole,10,Omeprazole Delayed Release Capsule,"CAPSULE, DELAYED RELEASE",51,63632501.0,28344703
5,hydrochlorothiazide,Hydrochlorothiazide,2,Hydrochlorothiazide Capsule,"CAPSULE, GELATIN COATED",51,45875396.3,18114858
6,potassium chloride,Potassium Chloride,2,"Potassium Chloride Tablet, Extended Release","TABLET, EXTENDED RELEASE",51,29802722.5,15738303
7,montelukast sodium,montelukast sodium,1,"Montelukast Sodium Tablet, Chewable","TABLET, CHEWABLE",51,27062068.2,12090298
8,albuterol sulfate,Albuterol Sulfate,3,Albuterol Sulfate Solution,SOLUTION,51,25008428.8,20974090
9,allopurinol,Allopurinol,3,Allopurinol Tablet,TABLET,51,22959134.2,9582901


In [16]:
# ============================================================
# PHARMAOPT — CASE STUDY 1
# LISDEXAMFETAMINE DIMESYLATE
# ============================================================

SELECTED_DRUG = "lisdexamfetamine dimesylate"

drug_demand = (
    generic_state_demand_2024[
        generic_state_demand_2024[
            "generic_normalized"
        ].eq(SELECTED_DRUG)
    ]
    .copy()
)

# ------------------------------------------------------------
# CALCULATE GEOGRAPHIC DEMAND SHARES
# ------------------------------------------------------------

total_demand_proxy = (
    drug_demand[
        "total_30day_fills"
    ].sum()
)

drug_demand["demand_share"] = (
    drug_demand["total_30day_fills"]
    / total_demand_proxy
)

drug_demand = (
    drug_demand
    .sort_values(
        "total_30day_fills",
        ascending=False
    )
    .reset_index(drop=True)
)


print("PHARMAOPT — CASE STUDY 1")
print("=" * 75)

print(
    "Drug:",
    drug_demand["generic_name"].iloc[0]
)

print(
    "Geographic markets:",
    drug_demand["state"].nunique()
)

print(
    "Total 2024 CMS 30-day-fill demand proxy:",
    f"{total_demand_proxy:,.1f}"
)

print(
    "Demand-share total:",
    f"{drug_demand['demand_share'].sum():.6f}"
)

print("\nTOP 15 MARKETS BY DEMAND")
print("=" * 75)

display(
    drug_demand[
        [
            "state",
            "total_30day_fills",
            "total_claims",
            "demand_share"
        ]
    ]
    .head(15)
)

PHARMAOPT — CASE STUDY 1
Drug: Lisdexamfetamine Dimesylate
Geographic markets: 51
Total 2024 CMS 30-day-fill demand proxy: 434,639.5
Demand-share total: 1.000000

TOP 15 MARKETS BY DEMAND


,state,total_30day_fills,total_claims,demand_share
0,Texas,26408.6,23726,0.060760
1,California,23232.4,20836,0.053452
2,Wisconsin,23082.6,22588,0.053107
3,Florida,23049.2,21188,0.053031
4,New York,22349.1,20626,0.051420
5,North Carolina,19214.7,18249,0.044208
6,Ohio,18065.5,17095,0.041564
7,Indiana,14876.6,14110,0.034227
8,Pennsylvania,14168.3,13255,0.032598
9,Massachusetts,13853.3,13263,0.031873


In [17]:
# ============================================================
# PHARMAOPT — MODELED SUPPLY CONSTRAINT
# ============================================================

SUPPLY_RATIO = 0.70

modeled_available_supply = (
    total_demand_proxy
    * SUPPLY_RATIO
)

modeled_supply_deficit = (
    total_demand_proxy
    - modeled_available_supply
)

drug_demand["modeled_full_demand"] = (
    drug_demand["total_30day_fills"]
)

print("MODELED SUPPLY-CONSTRAINED SCENARIO")
print("=" * 75)

print(
    "Demand proxy:",
    f"{total_demand_proxy:,.1f}"
)

print(
    "Available supply assumption:",
    f"{SUPPLY_RATIO:.0%}"
)

print(
    "Modeled available supply:",
    f"{modeled_available_supply:,.1f}"
)

print(
    "Modeled supply deficit:",
    f"{modeled_supply_deficit:,.1f}"
)

print(
    "Modeled shortage severity:",
    f"{1 - SUPPLY_RATIO:.0%}"
)

MODELED SUPPLY-CONSTRAINED SCENARIO
Demand proxy: 434,639.5
Available supply assumption: 70%
Modeled available supply: 304,247.6
Modeled supply deficit: 130,391.8
Modeled shortage severity: 30%


In [18]:
# ============================================================
# PHARMAOPT — FIX + VALIDATE NDC ↔ CMS GENERIC BRIDGE
# ============================================================

# One row per NORMALIZED NDC generic name.
# This prevents capitalization/spelling variants that normalize
# to the same value from duplicating CMS demand rows.

unique_ndc_generics = (
    shortage_ndc_products[
        [
            "ndc_generic_normalized",
            "ndc_generic_name"
        ]
    ]
    .dropna(subset=["ndc_generic_normalized"])
    .sort_values("ndc_generic_name")
    .drop_duplicates(
        subset=["ndc_generic_normalized"],
        keep="first"
    )
    .copy()
)


cms_ndc_overlap_clean = (
    generic_state_demand_2024.merge(
        unique_ndc_generics,
        left_on="generic_normalized",
        right_on="ndc_generic_normalized",
        how="inner",
        validate="many_to_one"
    )
)


clean_match_coverage = (
    cms_ndc_overlap_clean
    .groupby(
        [
            "generic_normalized",
            "generic_name"
        ],
        as_index=False
    )
    .agg(
        states_with_demand=("state", "nunique"),
        total_30day_fills=("total_30day_fills", "sum"),
        total_claims=("total_claims", "sum")
    )
    .sort_values(
        [
            "states_with_demand",
            "total_30day_fills"
        ],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)


print("CLEAN FDA NDC ↔ CMS BRIDGE")
print("=" * 75)

print(
    "Unique matched generic drugs:",
    f"{clean_match_coverage['generic_normalized'].nunique():,}"
)

print(
    "Matched drug-state observations:",
    f"{len(cms_ndc_overlap_clean):,}"
)

print(
    "Maximum geographic coverage:",
    clean_match_coverage["states_with_demand"].max()
)


# ------------------------------------------------------------
# VALIDATE CASE STUDY
# ------------------------------------------------------------

case_validation = clean_match_coverage[
    clean_match_coverage[
        "generic_normalized"
    ].eq(SELECTED_DRUG)
]

print("\nCASE STUDY VALIDATION")
print("=" * 75)

display(case_validation)

clean_case_total = (
    case_validation["total_30day_fills"].iloc[0]
)

direct_case_total = (
    drug_demand["total_30day_fills"].sum()
)

print(
    "Clean bridge demand:",
    f"{clean_case_total:,.1f}"
)

print(
    "Direct demand-table demand:",
    f"{direct_case_total:,.1f}"
)

print(
    "Difference:",
    f"{clean_case_total - direct_case_total:,.6f}"
)

assert np.isclose(
    clean_case_total,
    direct_case_total
)

print("\nVALIDATION PASSED — no duplicated demand.")

CLEAN FDA NDC ↔ CMS BRIDGE
Unique matched generic drugs: 107
Matched drug-state observations: 4,344
Maximum geographic coverage: 51

CASE STUDY VALIDATION


,generic_normalized,generic_name,states_with_demand,total_30day_fills,total_claims
38,lisdexamfetamine dimesylate,Lisdexamfetamine Dimesylate,51,434639.5,409425


Clean bridge demand: 434,639.5
Direct demand-table demand: 434,639.5
Difference: 0.000000

VALIDATION PASSED — no duplicated demand.


In [19]:
# ============================================================
# PHARMAOPT — PROPORTIONAL ALLOCATION BASELINE
# ============================================================

allocation_model = drug_demand[
    [
        "state",
        "total_30day_fills",
        "total_claims",
        "demand_share"
    ]
].copy()

allocation_model = allocation_model.rename(
    columns={
        "total_30day_fills": "demand"
    }
)

# Proportional baseline:
# every market receives the same percentage of its demand.
allocation_model["proportional_allocation"] = (
    allocation_model["demand"]
    * SUPPLY_RATIO
)

allocation_model["proportional_unmet"] = (
    allocation_model["demand"]
    - allocation_model["proportional_allocation"]
)

allocation_model["proportional_service_rate"] = (
    allocation_model["proportional_allocation"]
    / allocation_model["demand"]
)

print("PROPORTIONAL ALLOCATION BASELINE")
print("=" * 75)

print(
    "Total allocated:",
    f"{allocation_model['proportional_allocation'].sum():,.1f}"
)

print(
    "Total unmet demand:",
    f"{allocation_model['proportional_unmet'].sum():,.1f}"
)

print(
    "Minimum service rate:",
    f"{allocation_model['proportional_service_rate'].min():.1%}"
)

print(
    "Maximum service rate:",
    f"{allocation_model['proportional_service_rate'].max():.1%}"
)

display(
    allocation_model[
        [
            "state",
            "demand",
            "demand_share",
            "proportional_allocation",
            "proportional_unmet",
            "proportional_service_rate"
        ]
    ].head(15)
)

PROPORTIONAL ALLOCATION BASELINE
Total allocated: 304,247.6
Total unmet demand: 130,391.9
Minimum service rate: 70.0%
Maximum service rate: 70.0%


,state,demand,demand_share,proportional_allocation,proportional_unmet,proportional_service_rate
0,Texas,26408.6,0.060760,18486.02,7922.58,0.7
1,California,23232.4,0.053452,16262.68,6969.72,0.7
2,Wisconsin,23082.6,0.053107,16157.82,6924.78,0.7
3,Florida,23049.2,0.053031,16134.44,6914.76,0.7
4,New York,22349.1,0.051420,15644.37,6704.73,0.7
5,North Carolina,19214.7,0.044208,13450.29,5764.41,0.7
6,Ohio,18065.5,0.041564,12645.85,5419.65,0.7
7,Indiana,14876.6,0.034227,10413.62,4462.98,0.7
8,Pennsylvania,14168.3,0.032598,9917.81,4250.49,0.7
9,Massachusetts,13853.3,0.031873,9697.31,4155.99,0.7


In [20]:
# ============================================================
# PHARMAOPT — OR-TOOLS OPTIMIZATION MODEL
# ============================================================

from ortools.linear_solver import pywraplp

MIN_SERVICE_RATE = 0.60

solver = pywraplp.Solver.CreateSolver("GLOP")

if solver is None:
    raise RuntimeError("OR-Tools GLOP solver could not be created.")


# ------------------------------------------------------------
# DECISION VARIABLES
# ------------------------------------------------------------

allocation_vars = {}
unmet_vars = {}

mean_demand = allocation_model["demand"].mean()


for row in allocation_model.itertuples():

    state = row.state
    demand = float(row.demand)

    minimum_allocation = (
        MIN_SERVICE_RATE * demand
    )

    allocation_vars[state] = solver.NumVar(
        minimum_allocation,
        demand,
        f"allocation_{state}"
    )

    unmet_vars[state] = solver.NumVar(
        0,
        demand,
        f"unmet_{state}"
    )

    # Demand = allocation + unmet demand
    solver.Add(
        allocation_vars[state]
        + unmet_vars[state]
        == demand
    )


# ------------------------------------------------------------
# TOTAL SUPPLY CONSTRAINT
# ------------------------------------------------------------

solver.Add(
    solver.Sum(
        allocation_vars.values()
    )
    <= modeled_available_supply
)


# ------------------------------------------------------------
# OBJECTIVE
#
# Higher-demand markets receive greater priority,
# while the 60% service floor protects every market.
# ------------------------------------------------------------

objective_terms = []

for row in allocation_model.itertuples():

    state = row.state
    demand = float(row.demand)

    demand_pressure_weight = (
        demand / mean_demand
    )

    objective_terms.append(
        demand_pressure_weight
        * unmet_vars[state]
    )


solver.Minimize(
    solver.Sum(objective_terms)
)


# ------------------------------------------------------------
# SOLVE
# ------------------------------------------------------------

status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:

    print("OPTIMAL SOLUTION FOUND")

elif status == pywraplp.Solver.FEASIBLE:

    print("FEASIBLE SOLUTION FOUND")

else:

    raise RuntimeError(
        f"Optimization failed. Solver status: {status}"
    )

OPTIMAL SOLUTION FOUND


In [21]:
# ============================================================
# PHARMAOPT — EXTRACT OPTIMIZED ALLOCATION
# ============================================================

allocation_model["optimized_allocation"] = (
    allocation_model["state"]
    .map(
        lambda state:
        allocation_vars[state].solution_value()
    )
)

allocation_model["optimized_unmet"] = (
    allocation_model["demand"]
    - allocation_model["optimized_allocation"]
)

allocation_model["optimized_service_rate"] = (
    allocation_model["optimized_allocation"]
    / allocation_model["demand"]
)


print("PHARMAOPT OPTIMIZED ALLOCATION")
print("=" * 75)

print(
    "Total available supply:",
    f"{modeled_available_supply:,.1f}"
)

print(
    "Total optimized allocation:",
    f"{allocation_model['optimized_allocation'].sum():,.1f}"
)

print(
    "Total unmet demand:",
    f"{allocation_model['optimized_unmet'].sum():,.1f}"
)

print(
    "Minimum service rate:",
    f"{allocation_model['optimized_service_rate'].min():.1%}"
)

print(
    "Maximum service rate:",
    f"{allocation_model['optimized_service_rate'].max():.1%}"
)


# ------------------------------------------------------------
# VALIDATION CHECKS
# ------------------------------------------------------------

assert (
    allocation_model["optimized_allocation"]
    <= allocation_model["demand"] + 1e-6
).all()

assert (
    allocation_model["optimized_service_rate"]
    >= MIN_SERVICE_RATE - 1e-6
).all()

assert (
    allocation_model["optimized_allocation"].sum()
    <= modeled_available_supply + 1e-6
)

print("\nOPTIMIZATION VALIDATION PASSED.")


# ------------------------------------------------------------
# COMPARE WITH PROPORTIONAL BASELINE
# ------------------------------------------------------------

allocation_model["allocation_change"] = (
    allocation_model["optimized_allocation"]
    - allocation_model["proportional_allocation"]
)

display(
    allocation_model[
        [
            "state",
            "demand",
            "proportional_allocation",
            "optimized_allocation",
            "allocation_change",
            "optimized_unmet",
            "optimized_service_rate"
        ]
    ]
    .sort_values(
        "demand",
        ascending=False
    )
    .head(20)
)

PHARMAOPT OPTIMIZED ALLOCATION
Total available supply: 304,247.6
Total optimized allocation: 304,247.6
Total unmet demand: 130,391.9
Minimum service rate: 60.0%
Maximum service rate: 100.0%

OPTIMIZATION VALIDATION PASSED.


,state,demand,proportional_allocation,optimized_allocation,allocation_change,optimized_unmet,optimized_service_rate
0,Texas,26408.6,18486.02,26408.60,7922.58,0.00,1.00000
1,California,23232.4,16262.68,23232.40,6969.72,0.00,1.00000
2,Wisconsin,23082.6,16157.82,23082.60,6924.78,0.00,1.00000
3,Florida,23049.2,16134.44,23049.20,6914.76,0.00,1.00000
4,New York,22349.1,15644.37,18564.29,2919.92,3784.81,0.83065
5,North Carolina,19214.7,13450.29,11528.82,-1921.47,7685.88,0.60000
6,Ohio,18065.5,12645.85,10839.30,-1806.55,7226.20,0.60000
7,Indiana,14876.6,10413.62,8925.96,-1487.66,5950.64,0.60000
8,Pennsylvania,14168.3,9917.81,8500.98,-1416.83,5667.32,0.60000
9,Massachusetts,13853.3,9697.31,8311.98,-1385.33,5541.32,0.60000


In [22]:
# ============================================================
# PHARMAOPT — EVALUATE OPTIMIZATION POLICY V0.1
# ============================================================

demand_weights = (
    allocation_model["demand"]
    / allocation_model["demand"].mean()
)

baseline_weighted_unmet = (
    demand_weights
    * allocation_model["proportional_unmet"]
).sum()

optimized_weighted_unmet = (
    demand_weights
    * allocation_model["optimized_unmet"]
).sum()


policy_evaluation = pd.DataFrame({
    "metric": [
        "Total allocation",
        "Total unmet demand",
        "Weighted unmet-demand score",
        "Minimum service rate",
        "Maximum service rate",
        "Service-rate standard deviation",
        "Markets at minimum service",
        "Markets at full service"
    ],

    "Proportional Baseline": [
        allocation_model["proportional_allocation"].sum(),
        allocation_model["proportional_unmet"].sum(),
        baseline_weighted_unmet,
        allocation_model["proportional_service_rate"].min(),
        allocation_model["proportional_service_rate"].max(),
        allocation_model["proportional_service_rate"].std(),
        np.isclose(
            allocation_model["proportional_service_rate"],
            MIN_SERVICE_RATE
        ).sum(),
        np.isclose(
            allocation_model["proportional_service_rate"],
            1.0
        ).sum()
    ],

    "Demand-Priority Optimization": [
        allocation_model["optimized_allocation"].sum(),
        allocation_model["optimized_unmet"].sum(),
        optimized_weighted_unmet,
        allocation_model["optimized_service_rate"].min(),
        allocation_model["optimized_service_rate"].max(),
        allocation_model["optimized_service_rate"].std(),
        np.isclose(
            allocation_model["optimized_service_rate"],
            MIN_SERVICE_RATE
        ).sum(),
        np.isclose(
            allocation_model["optimized_service_rate"],
            1.0
        ).sum()
    ]
})

display(policy_evaluation)

weighted_improvement = (
    (
        baseline_weighted_unmet
        - optimized_weighted_unmet
    )
    / baseline_weighted_unmet
)

print(
    "\nReduction in demand-weighted unmet score:",
    f"{weighted_improvement:.2%}"
)

,metric,Proportional Baseline,Demand-Priority Optimization
0,Total allocation,3.042476e+05,304247.650000
1,Total unmet demand,1.303919e+05,130391.850000
2,Weighted unmet-demand score,2.121283e+05,161310.410368
3,Minimum service rate,7.000000e-01,0.600000
4,Maximum service rate,7.000000e-01,1.000000
5,Service-rate standard deviation,1.164412e-16,0.112026
6,Markets at minimum service,0.000000e+00,46.000000
7,Markets at full service,0.000000e+00,4.000000



Reduction in demand-weighted unmet score: 23.96%


In [23]:
# ============================================================
# PHARMAOPT OPTIMIZATION V0.2
# BALANCED DEMAND-PRIORITY ALLOCATION
# ============================================================

BALANCED_MIN_SERVICE = 0.60
BALANCED_MAX_SERVICE = 0.80

balanced_solver = pywraplp.Solver.CreateSolver("GLOP")

if balanced_solver is None:
    raise RuntimeError(
        "OR-Tools GLOP solver could not be created."
    )


balanced_allocation_vars = {}
balanced_unmet_vars = {}

mean_demand = allocation_model["demand"].mean()


# ------------------------------------------------------------
# DECISION VARIABLES + MARKET CONSTRAINTS
# ------------------------------------------------------------

for row in allocation_model.itertuples():

    state = row.state
    demand = float(row.demand)

    minimum_allocation = (
        BALANCED_MIN_SERVICE * demand
    )

    maximum_allocation = (
        BALANCED_MAX_SERVICE * demand
    )

    balanced_allocation_vars[state] = (
        balanced_solver.NumVar(
            minimum_allocation,
            maximum_allocation,
            f"balanced_allocation_{state}"
        )
    )

    balanced_unmet_vars[state] = (
        balanced_solver.NumVar(
            0,
            demand,
            f"balanced_unmet_{state}"
        )
    )

    balanced_solver.Add(
        balanced_allocation_vars[state]
        + balanced_unmet_vars[state]
        == demand
    )


# ------------------------------------------------------------
# TOTAL AVAILABLE SUPPLY
# ------------------------------------------------------------

balanced_solver.Add(
    balanced_solver.Sum(
        balanced_allocation_vars.values()
    )
    <= modeled_available_supply
)


# ------------------------------------------------------------
# DEMAND-PRESSURE OBJECTIVE
# ------------------------------------------------------------

balanced_objective_terms = []

for row in allocation_model.itertuples():

    state = row.state
    demand = float(row.demand)

    demand_pressure_weight = (
        demand / mean_demand
    )

    balanced_objective_terms.append(
        demand_pressure_weight
        * balanced_unmet_vars[state]
    )


balanced_solver.Minimize(
    balanced_solver.Sum(
        balanced_objective_terms
    )
)


# ------------------------------------------------------------
# SOLVE
# ------------------------------------------------------------

balanced_status = balanced_solver.Solve()

if balanced_status == pywraplp.Solver.OPTIMAL:
    print("BALANCED OPTIMAL SOLUTION FOUND")

elif balanced_status == pywraplp.Solver.FEASIBLE:
    print("BALANCED FEASIBLE SOLUTION FOUND")

else:
    raise RuntimeError(
        f"Balanced optimization failed. "
        f"Status: {balanced_status}"
    )

BALANCED OPTIMAL SOLUTION FOUND


In [24]:
# ============================================================
# PHARMAOPT — EXTRACT BALANCED OPTIMIZATION RESULTS
# ============================================================

allocation_model["balanced_allocation"] = (
    allocation_model["state"].map(
        lambda state:
        balanced_allocation_vars[
            state
        ].solution_value()
    )
)

allocation_model["balanced_unmet"] = (
    allocation_model["demand"]
    - allocation_model["balanced_allocation"]
)

allocation_model["balanced_service_rate"] = (
    allocation_model["balanced_allocation"]
    / allocation_model["demand"]
)


print("PHARMAOPT BALANCED OPTIMIZATION")
print("=" * 75)

print(
    "Total available supply:",
    f"{modeled_available_supply:,.1f}"
)

print(
    "Total allocated:",
    f"{allocation_model['balanced_allocation'].sum():,.1f}"
)

print(
    "Total unmet demand:",
    f"{allocation_model['balanced_unmet'].sum():,.1f}"
)

print(
    "Minimum service rate:",
    f"{allocation_model['balanced_service_rate'].min():.1%}"
)

print(
    "Maximum service rate:",
    f"{allocation_model['balanced_service_rate'].max():.1%}"
)


assert (
    allocation_model["balanced_service_rate"]
    >= BALANCED_MIN_SERVICE - 1e-6
).all()

assert (
    allocation_model["balanced_service_rate"]
    <= BALANCED_MAX_SERVICE + 1e-6
).all()

assert (
    allocation_model["balanced_allocation"].sum()
    <= modeled_available_supply + 1e-6
)

print("\nBALANCED OPTIMIZATION VALIDATION PASSED.")


display(
    allocation_model[
        [
            "state",
            "demand",
            "proportional_service_rate",
            "optimized_service_rate",
            "balanced_service_rate",
            "balanced_allocation",
            "balanced_unmet"
        ]
    ]
    .sort_values(
        "demand",
        ascending=False
    )
    .head(20)
)

PHARMAOPT BALANCED OPTIMIZATION
Total available supply: 304,247.6
Total allocated: 304,247.6
Total unmet demand: 130,391.9
Minimum service rate: 60.0%
Maximum service rate: 80.0%

BALANCED OPTIMIZATION VALIDATION PASSED.


,state,demand,proportional_service_rate,optimized_service_rate,balanced_service_rate,balanced_allocation,balanced_unmet
0,Texas,26408.6,0.7,1.00000,0.800000,21126.88,5281.72
1,California,23232.4,0.7,1.00000,0.800000,18585.92,4646.48
2,Wisconsin,23082.6,0.7,1.00000,0.800000,18466.08,4616.52
3,Florida,23049.2,0.7,1.00000,0.800000,18439.36,4609.84
4,New York,22349.1,0.7,0.83065,0.800000,17879.28,4469.82
5,North Carolina,19214.7,0.7,0.60000,0.800000,15371.76,3842.94
6,Ohio,18065.5,0.7,0.60000,0.800000,14452.40,3613.10
7,Indiana,14876.6,0.7,0.60000,0.800000,11901.28,2975.32
8,Pennsylvania,14168.3,0.7,0.60000,0.800000,11334.64,2833.66
9,Massachusetts,13853.3,0.7,0.60000,0.800000,11082.64,2770.66


In [25]:
# ============================================================
# PHARMAOPT — POLICY COMPARISON
# ============================================================

demand_weights = (
    allocation_model["demand"]
    / allocation_model["demand"].mean()
)

# Weighted unmet-demand scores
proportional_weighted_unmet = (
    demand_weights
    * allocation_model["proportional_unmet"]
).sum()

priority_weighted_unmet = (
    demand_weights
    * allocation_model["optimized_unmet"]
).sum()

balanced_weighted_unmet = (
    demand_weights
    * allocation_model["balanced_unmet"]
).sum()


policy_comparison = pd.DataFrame({

    "policy": [
        "Proportional",
        "Demand-Priority",
        "Balanced PharmaOpt"
    ],

    "total_allocated": [
        allocation_model["proportional_allocation"].sum(),
        allocation_model["optimized_allocation"].sum(),
        allocation_model["balanced_allocation"].sum()
    ],

    "total_unmet": [
        allocation_model["proportional_unmet"].sum(),
        allocation_model["optimized_unmet"].sum(),
        allocation_model["balanced_unmet"].sum()
    ],

    "weighted_unmet_score": [
        proportional_weighted_unmet,
        priority_weighted_unmet,
        balanced_weighted_unmet
    ],

    "minimum_service_rate": [
        allocation_model["proportional_service_rate"].min(),
        allocation_model["optimized_service_rate"].min(),
        allocation_model["balanced_service_rate"].min()
    ],

    "maximum_service_rate": [
        allocation_model["proportional_service_rate"].max(),
        allocation_model["optimized_service_rate"].max(),
        allocation_model["balanced_service_rate"].max()
    ],

    "service_rate_std": [
        allocation_model["proportional_service_rate"].std(),
        allocation_model["optimized_service_rate"].std(),
        allocation_model["balanced_service_rate"].std()
    ]
})


# Improvement relative to proportional baseline
policy_comparison[
    "weighted_score_improvement_vs_proportional"
] = (
    (
        proportional_weighted_unmet
        - policy_comparison["weighted_unmet_score"]
    )
    / proportional_weighted_unmet
)


print("PHARMAOPT — ALLOCATION POLICY COMPARISON")
print("=" * 80)

display(policy_comparison.round(4))

PHARMAOPT — ALLOCATION POLICY COMPARISON


,policy,total_allocated,total_unmet,weighted_unmet_score,minimum_service_rate,maximum_service_rate,service_rate_std,weighted_score_improvement_vs_proportional
0,Proportional,304247.65,130391.85,212128.3446,0.7,0.7,0.0000,0.0000
1,Demand-Priority,304247.65,130391.85,161310.4104,0.6,1.0,0.1120,0.2396
2,Balanced PharmaOpt,304247.65,130391.85,180587.8828,0.6,0.8,0.0831,0.1487


In [26]:
# ============================================================
# PHARMAOPT — MARKET-LEVEL ALLOCATION IMPACT
# ============================================================

market_impact = allocation_model[
    [
        "state",
        "demand",
        "proportional_allocation",
        "balanced_allocation",
        "proportional_service_rate",
        "balanced_service_rate"
    ]
].copy()


market_impact["allocation_change"] = (
    market_impact["balanced_allocation"]
    - market_impact["proportional_allocation"]
)

market_impact["allocation_change_pct_of_demand"] = (
    market_impact["allocation_change"]
    / market_impact["demand"]
)


print("MARKETS RECEIVING MORE THAN PROPORTIONAL BASELINE")
print("=" * 80)

display(
    market_impact
    .sort_values(
        "allocation_change",
        ascending=False
    )
    .head(15)
)


print("\nMARKETS RECEIVING LESS THAN PROPORTIONAL BASELINE")
print("=" * 80)

display(
    market_impact
    .sort_values(
        "allocation_change",
        ascending=True
    )
    .head(15)
)

MARKETS RECEIVING MORE THAN PROPORTIONAL BASELINE


,state,demand,proportional_allocation,balanced_allocation,proportional_service_rate,balanced_service_rate,allocation_change,allocation_change_pct_of_demand
0,Texas,26408.6,18486.02,21126.88,0.7,0.800000,2640.86,0.100000
1,California,23232.4,16262.68,18585.92,0.7,0.800000,2323.24,0.100000
2,Wisconsin,23082.6,16157.82,18466.08,0.7,0.800000,2308.26,0.100000
3,Florida,23049.2,16134.44,18439.36,0.7,0.800000,2304.92,0.100000
4,New York,22349.1,15644.37,17879.28,0.7,0.800000,2234.91,0.100000
5,North Carolina,19214.7,13450.29,15371.76,0.7,0.800000,1921.47,0.100000
6,Ohio,18065.5,12645.85,14452.40,0.7,0.800000,1806.55,0.100000
7,Indiana,14876.6,10413.62,11901.28,0.7,0.800000,1487.66,0.100000
8,Pennsylvania,14168.3,9917.81,11334.64,0.7,0.800000,1416.83,0.100000
9,Massachusetts,13853.3,9697.31,11082.64,0.7,0.800000,1385.33,0.100000



MARKETS RECEIVING LESS THAN PROPORTIONAL BASELINE


,state,demand,proportional_allocation,balanced_allocation,proportional_service_rate,balanced_service_rate,allocation_change,allocation_change_pct_of_demand
12,Illinois,11901.5,8331.05,7140.90,0.7,0.6,-1190.15,-0.1
13,Georgia,11388.0,7971.60,6832.80,0.7,0.6,-1138.80,-0.1
14,Maine,9829.5,6880.65,5897.70,0.7,0.6,-982.95,-0.1
15,Virginia,9692.2,6784.54,5815.32,0.7,0.6,-969.22,-0.1
16,Louisiana,9546.6,6682.62,5727.96,0.7,0.6,-954.66,-0.1
17,Connecticut,9128.6,6390.02,5477.16,0.7,0.6,-912.86,-0.1
18,Maryland,9103.9,6372.73,5462.34,0.7,0.6,-910.39,-0.1
19,Alabama,9048.0,6333.60,5428.80,0.7,0.6,-904.80,-0.1
20,Washington,9037.0,6325.90,5422.20,0.7,0.6,-903.70,-0.1
21,Missouri,8895.6,6226.92,5337.36,0.7,0.6,-889.56,-0.1


In [27]:
# ============================================================
# PHARMAOPT — SUPPLY-SHOCK SENSITIVITY ANALYSIS
# ============================================================

def run_balanced_allocation(
    demand_df,
    supply_ratio,
    service_band=0.10
):

    scenario_solver = (
        pywraplp.Solver.CreateSolver("GLOP")
    )

    if scenario_solver is None:
        raise RuntimeError(
            "Could not create OR-Tools solver."
        )

    # Service guardrails move with supply availability
    min_service = max(
        0.0,
        supply_ratio - service_band
    )

    max_service = min(
        1.0,
        supply_ratio + service_band
    )

    total_demand = demand_df["demand"].sum()

    available_supply = (
        total_demand
        * supply_ratio
    )

    mean_demand = demand_df["demand"].mean()

    allocation_vars = {}
    unmet_vars = {}


    # --------------------------------------------------------
    # VARIABLES + CONSTRAINTS
    # --------------------------------------------------------

    for row in demand_df.itertuples():

        state = row.state
        demand = float(row.demand)

        allocation_vars[state] = (
            scenario_solver.NumVar(
                min_service * demand,
                max_service * demand,
                f"allocation_{state}"
            )
        )

        unmet_vars[state] = (
            scenario_solver.NumVar(
                0,
                demand,
                f"unmet_{state}"
            )
        )

        scenario_solver.Add(
            allocation_vars[state]
            + unmet_vars[state]
            == demand
        )


    # --------------------------------------------------------
    # SUPPLY CONSTRAINT
    # --------------------------------------------------------

    scenario_solver.Add(
        scenario_solver.Sum(
            allocation_vars.values()
        )
        <= available_supply
    )


    # --------------------------------------------------------
    # DEMAND-PRESSURE OBJECTIVE
    # --------------------------------------------------------

    objective_terms = []

    for row in demand_df.itertuples():

        state = row.state
        demand = float(row.demand)

        weight = demand / mean_demand

        objective_terms.append(
            weight
            * unmet_vars[state]
        )


    scenario_solver.Minimize(
        scenario_solver.Sum(
            objective_terms
        )
    )


    status = scenario_solver.Solve()

    if status != pywraplp.Solver.OPTIMAL:
        raise RuntimeError(
            f"Scenario failed at supply ratio "
            f"{supply_ratio:.0%}"
        )


    allocations = np.array([
        allocation_vars[state].solution_value()
        for state in demand_df["state"]
    ])

    demands = demand_df["demand"].to_numpy()

    unmet = demands - allocations

    service_rates = allocations / demands

    weights = (
        demands
        / demands.mean()
    )

    weighted_unmet = (
        weights * unmet
    ).sum()


    return {
        "supply_ratio": supply_ratio,
        "shortage_severity": 1 - supply_ratio,
        "available_supply": available_supply,
        "total_unmet": unmet.sum(),
        "weighted_unmet_score": weighted_unmet,
        "minimum_service_rate": service_rates.min(),
        "maximum_service_rate": service_rates.max(),
        "service_rate_std": service_rates.std()
    }


# ------------------------------------------------------------
# RUN MULTIPLE SUPPLY CONDITIONS
# ------------------------------------------------------------

supply_scenarios = [
    0.50,
    0.60,
    0.70,
    0.80,
    0.90
]

sensitivity_results = []

for ratio in supply_scenarios:

    result = run_balanced_allocation(
        allocation_model,
        supply_ratio=ratio,
        service_band=0.10
    )

    sensitivity_results.append(result)


sensitivity_df = pd.DataFrame(
    sensitivity_results
)


print("PHARMAOPT — SUPPLY-SHOCK SENSITIVITY ANALYSIS")
print("=" * 80)

display(
    sensitivity_df.round(4)
)

PHARMAOPT — SUPPLY-SHOCK SENSITIVITY ANALYSIS


,supply_ratio,shortage_severity,available_supply,total_unmet,weighted_unmet_score,minimum_service_rate,maximum_service_rate,service_rate_std
0,0.5,0.5,217319.75,217319.75,322006.7792,0.4,0.6,0.0823
1,0.6,0.4,260783.70,173855.80,251297.3310,0.5,0.7,0.0823
2,0.7,0.3,304247.65,130391.85,180587.8828,0.6,0.8,0.0823
3,0.8,0.2,347711.60,86927.90,109878.4346,0.7,0.9,0.0823
4,0.9,0.1,391175.55,43463.95,39168.9864,0.8,1.0,0.0823


In [28]:
# ============================================================
# PHARMAOPT — ROBUSTNESS VS PROPORTIONAL BASELINE
# ============================================================

robustness_results = []

for row in sensitivity_df.itertuples():

    supply_ratio = row.supply_ratio

    # --------------------------------------------------------
    # PROPORTIONAL BASELINE FOR THIS SUPPLY LEVEL
    # --------------------------------------------------------

    proportional_allocation = (
        allocation_model["demand"]
        * supply_ratio
    )

    proportional_unmet = (
        allocation_model["demand"]
        - proportional_allocation
    )

    weights = (
        allocation_model["demand"]
        / allocation_model["demand"].mean()
    )

    proportional_weighted_score = (
        weights
        * proportional_unmet
    ).sum()

    # --------------------------------------------------------
    # PHARMAOPT RESULT
    # --------------------------------------------------------

    pharmaopt_weighted_score = (
        row.weighted_unmet_score
    )

    improvement = (
        (
            proportional_weighted_score
            - pharmaopt_weighted_score
        )
        / proportional_weighted_score
    )

    robustness_results.append({
        "supply_ratio": supply_ratio,
        "shortage_severity": 1 - supply_ratio,
        "proportional_weighted_score":
            proportional_weighted_score,
        "pharmaopt_weighted_score":
            pharmaopt_weighted_score,
        "weighted_score_improvement":
            improvement
    })


robustness_df = pd.DataFrame(
    robustness_results
)


print("PHARMAOPT — ROBUSTNESS VS PROPORTIONAL BASELINE")
print("=" * 80)

display(
    robustness_df.round(4)
)


print("\nIMPROVEMENT RANGE")
print("=" * 80)

print(
    "Minimum improvement:",
    f"{robustness_df['weighted_score_improvement'].min():.2%}"
)

print(
    "Maximum improvement:",
    f"{robustness_df['weighted_score_improvement'].max():.2%}"
)

PHARMAOPT — ROBUSTNESS VS PROPORTIONAL BASELINE


,supply_ratio,shortage_severity,proportional_weighted_score,pharmaopt_weighted_score,weighted_score_improvement
0,0.5,0.5,353547.2411,322006.7792,0.0892
1,0.6,0.4,282837.7928,251297.3310,0.1115
2,0.7,0.3,212128.3446,180587.8828,0.1487
3,0.8,0.2,141418.8964,109878.4346,0.2230
4,0.9,0.1,70709.4482,39168.9864,0.4461



IMPROVEMENT RANGE
Minimum improvement: 8.92%
Maximum improvement: 44.61%


In [29]:
## Case Study 1 — Results and Interpretation

PharmaOpt evaluated geographic allocation strategies for **lisdexamfetamine dimesylate** using 2024 Medicare Part D utilization as a geographic demand proxy. FDA shortage records were connected to FDA NDC product records before exact normalized generic-name matching to CMS data. This produced a validated demand table covering all 50 states and Washington, D.C.

A modeled supply constraint of **70% of observed utilization demand** was used for the primary experiment. This represents a hypothetical stress-test scenario and does not represent observed manufacturer inventory.

### Allocation Policies

Three allocation strategies were evaluated:

- **Proportional allocation:** every market receives 70% of modeled demand.
- **Demand-priority optimization:** constrained optimization prioritizes higher-demand markets while guaranteeing a 60% minimum service level.
- **Balanced PharmaOpt:** demand-priority optimization with a 60% minimum and 80% maximum service-rate guardrail.

All three strategies distribute the same available supply and therefore produce the same total modeled unmet demand. The optimization problem instead determines **where the unavoidable shortage burden occurs**.

Under the 70% supply scenario, Balanced PharmaOpt reduced the demand-weighted unmet-demand score by approximately **14.87% relative to proportional allocation**, while maintaining service rates between 60% and 80%. The unconstrained demand-priority policy achieved a larger 23.96% improvement but produced service rates ranging from 60% to 100%, demonstrating a stronger efficiency-versus-fairness tradeoff.

### Sensitivity Analysis

Balanced PharmaOpt was also evaluated under modeled supply availability of 50%, 60%, 70%, 80%, and 90%. Relative to proportional allocation, the demand-weighted unmet-demand score improved in every tested scenario, with improvements ranging from approximately **8.92% to 44.61%**.

### Limitations

CMS Medicare Part D utilization is a demand proxy and does not represent total U.S. pharmaceutical demand. `Tot_30day_Fills` is a standardized utilization measure and is not interpreted as physical package inventory. The modeled supply constraints and service-rate guardrails are scenario assumptions rather than observed manufacturer inventory or clinical recommendations. FDA-to-CMS matching is performed at the generic-ingredient level and may not perfectly represent formulation-specific utilization.

This experiment demonstrates a decision-support optimization framework under explicit assumptions. It does not prescribe or autonomously execute real-world pharmaceutical distribution decisions.

SyntaxError: invalid syntax (200017229.py, line 3)